# Python Lesson 50: Conditional Probability, Bayes and Naive Bayes

**Concept page:** `wiki/concepts/conditional-probability-and-bayes.md` · **Area:** probability

**You will be able to:**
- apply Bayes' theorem to a diagnostic test
- build a tiny spam classifier by hand
- implement Gaussian Naive Bayes and compare with scikit-learn

*How to use:* run the cells top to bottom, read the output, then try the **Exercises** in the empty cells before opening the **Solutions** section at the bottom.

In [1]:
import numpy as np
import sympy as sp
import matplotlib
import matplotlib.pyplot as plt
np.set_printoptions(precision=4, suppress=True)
%matplotlib inline

## 1. Diagnostic test

In [2]:
sens,fpr,prev=0.99,0.05,0.01
p_pos=sens*prev+fpr*(1-prev); print(sens*prev/p_pos)

0.16666666666666669


## 2. Spam by word counts

In [3]:
emails=[("win lottery now",1),("lottery win win",1),("meeting at noon",0),("project meeting notes",0),("win a prize",1),("notes for noon",0)]
from collections import Counter
vocab=sorted({w for t,_ in emails for w in t.split()})
cnt={c:Counter(w for t,l in emails if l==c for w in t.split()) for c in (0,1)}
prior={c:sum(1 for _,l in emails if l==c)/len(emails) for c in (0,1)}
def score(text,c,alpha=1):
    tot=sum(cnt[c].values()); s=np.log(prior[c])
    for w in text.split(): s+=np.log((cnt[c][w]+alpha)/(tot+alpha*len(vocab)))
    return s
for t in ("win lottery","meeting notes"): print(t,"->","spam" if score(t,1)>score(t,0) else "ham")

win lottery -> spam
meeting notes -> ham


## 3. Gaussian Naive Bayes from scratch vs sklearn

In [4]:
rng=np.random.default_rng(0)
X=np.r_[rng.normal([20,5],[2,1],size=(100,2)),rng.normal([30,8],[3,1.5],size=(100,2))]; y=np.r_[np.zeros(100),np.ones(100)]
params={c:(X[y==c].mean(0),X[y==c].std(0,ddof=0),(y==c).mean()) for c in (0,1)}
def predict(Z):
    out=[]
    for z in Z:
        sc=[np.log(pr)+(-0.5*np.log(2*np.pi*s**2)-(z-m)**2/(2*s**2)).sum() for m,s,pr in (params[0],params[1])]
        out.append(int(np.argmax(sc)))
    return np.array(out)
mine=predict(X); print("accuracy",(mine==y).mean())
from sklearn.naive_bayes import GaussianNB
print("agrees with sklearn:",(GaussianNB().fit(X,y).predict(X)==mine).mean())

accuracy 0.995


agrees with sklearn: 1.0


## Cambodian textbook corner (កម្មវិធីសិក្សាកម្ពុជា)
Textbook link: Grade 9/Grade 11 probability uses P=favourable/possible and P(HHTT) as a product of independent probabilities — the 'naive' independence product here.

In [5]:
print(np.prod([2/3,2/3,1/3,1/3]))

0.04938271604938271


## Exercises

**Exercise 1.** Prevalence 0.1%, sensitivity 99%, false-positive 2%: P(disease|positive)? (about 0.047)

In [6]:
# your code here

**Exercise 2.** Classify 'win meeting' with the tiny spam model.

In [7]:
# your code here

## Solutions
*(Try the exercises first!)*

In [8]:
# Solution 1
p=0.99*0.001/(0.99*0.001+0.02*0.999); print(p); assert abs(p-0.0472)<1e-3

0.047210300429184546


In [9]:
# Solution 2
print('spam' if score('win meeting',1)>score('win meeting',0) else 'ham')

spam
